# Módulo 2 — Arquitetura de um *transformer*
**Objetivos de aprendizagem:** acompanhar o processamento de informações numa rede com camadas *transformer*
e demonstrar um LLM treinado a partir do zero.

| Tempo | Etapa |
|---|---|
| 10 min | Atenção do zero, em ~10 linhas de PyTorch |
| 5 min | Atenção dentro do BERTimbau real |
| 5 min | Tipos de arquitetura: encoder, decoder, encoder-decoder |
| 10 min (+ intervalo) | Mini-GPT treinado do zero em Machado de Assis |

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
MODO_RAPIDO = True   # True = subamostra para caber nos tempos da aula
SMOKE = os.environ.get("AULA_SMOKE") == "1"   # só para teste automatizado em CPU

if IN_COLAB:
    !pip install -q -U bertviz

import torch
import transformers
transformers.logging.set_verbosity_error()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| MODO_RAPIDO:", MODO_RAPIDO)
if DEVICE == "cpu" and not SMOKE:
    print("⚠️  Sem GPU. No Colab: Ambiente de execução → Alterar tipo → GPU T4.")

## 1. Atenção do zero
Cada token gera três vetores:
- **Q (query):** o que eu estou procurando?
- **K (key):** o que eu ofereço?
- **V (value):** a informação que eu passo adiante.

$$\text{Atenção}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

O produto $QK^\top$ mede a afinidade entre cada par de tokens. O softmax transforma isso em pesos que somam 1.
A saída de cada token é a **média ponderada dos V** de todos os tokens.

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

def atencao(Q, K, V, mascara=None):
    d_k = Q.size(-1)
    # TODO: calcule os scores (Q·Kᵀ / √d_k), aplique a máscara (posições com 0 viram -inf), faça o softmax e multiplique por V
    ...
    return saida, pesos

Um exemplo de brinquedo: 6 tokens, embeddings aleatórios de dimensão 16 e matrizes $W_Q, W_K, W_V$ aleatórias.
(Num modelo real essas matrizes são **aprendidas** no treino.)

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(0)
tokens = ["o", "cliente", "devolveu", "o", "produto", "quebrado"]
d = 16
X = torch.randn(len(tokens), d)                    # embeddings de entrada
W_q, W_k, W_v = (torch.randn(d, d) for _ in range(3))
Q, K, V = X @ W_q, X @ W_k, X @ W_v

saida, pesos = atencao(Q, K, V)
print("entrada:", tuple(X.shape), "→ saída:", tuple(saida.shape))

def mapa(pesos, rotulos_y, rotulos_x, titulo, eixo=None):
    eixo = eixo or plt.gca()
    eixo.imshow(pesos.detach(), cmap="Blues", vmin=0)
    eixo.set_xticks(range(len(rotulos_x)), rotulos_x, rotation=60)
    eixo.set_yticks(range(len(rotulos_y)), rotulos_y)
    eixo.set_title(titulo)

mascara_causal = torch.tril(torch.ones(len(tokens), len(tokens)))
_, pesos_causais = atencao(Q, K, V, mascara=mascara_causal)

fig, eixos = plt.subplots(1, 2, figsize=(10, 4))
mapa(pesos, tokens, tokens, "Bidirecional (estilo BERT)", eixos[0])
mapa(pesos_causais, tokens, tokens, "Causal (estilo GPT)", eixos[1])
plt.tight_layout()

**O que observar:**
- Cada **linha** é um token "olhando" para os outros, e soma 1.
- Na **máscara causal** um token só vê o passado. É isso que permite ao GPT gerar texto token a token.
- *Multi-head* = rodar várias atenções em paralelo, cada uma com seus próprios $W_Q, W_K, W_V$, e concatenar.

## 2. Atenção num modelo real: BERTimbau
Vamos pedir ao BERTimbau as matrizes de atenção de todas as 12 camadas × 12 cabeças e procurar
a cabeça que resolve o pronome **"ele" → "motorista"**.

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODELO = "neuralmind/bert-base-portuguese-cased"
tok = AutoTokenizer.from_pretrained(MODELO)
bert = AutoModel.from_pretrained(MODELO, attn_implementation="eager").eval()

frase = "O motorista prometeu entregar ontem, mas ele não cumpriu o prazo."
entrada = tok(frase, return_tensors="pt")
with torch.no_grad():
    atencoes = bert(**entrada, output_attentions=True).attentions   # 12 camadas × [1, 12 cabeças, T, T]
toks = tok.convert_ids_to_tokens(entrada.input_ids[0])
print(len(atencoes), "camadas |", tuple(atencoes[0].shape), "|", toks)

In [ ]:
i_ele, i_motorista = toks.index("ele"), toks.index("motorista")
# TODO: monte `afinidade` [camadas × cabeças] com o peso que "ele" dá a "motorista" e ache a camada/cabeça máxima
...
print(f"Camada {camada}, cabeça {cabeca}: 'ele' → 'motorista' com peso {afinidade[camada, cabeca]:.2f}")

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
eixos[0].imshow(afinidade, cmap="Reds")
eixos[0].set_xlabel("cabeça"); eixos[0].set_ylabel("camada")
eixos[0].set_title("Peso de 'ele' → 'motorista'")
mapa(atencoes[camada][0, cabeca], toks, toks, f"Camada {camada}, cabeça {cabeca}", eixos[1])
plt.tight_layout()

Exploração interativa: passe o mouse sobre os tokens e troque de camada.
Cabeças diferentes aprendem padrões diferentes (pontuação, token seguinte, sintaxe, correferência).

In [ ]:
from bertviz import head_view
head_view(atencoes, toks)

## 3. Três famílias de *transformer*

| | Encoder | Decoder | Encoder-decoder |
|---|---|---|---|
| Atenção | bidirecional | causal (só o passado) | encoder bidirecional + decoder causal com *cross-attention* |
| Treino | preencher palavras mascaradas | prever o próximo token | transformar sequência → sequência |
| Exemplos | BERT, BERTimbau, RoBERTa | GPT, Llama, Qwen, Gemini | T5, BART, Whisper |
| Bom para | classificar, NER, busca (embeddings) | gerar texto, chat, código | tradução, resumo |
| Nesta aula | **Módulo 1** | **mini-GPT abaixo, Módulo 3** | — |

A **ViT** (Módulo 4) é um encoder aplicado a pedaços de imagem.

## 4. Um LLM treinado do zero: mini-GPT
Vamos treinar um GPT **char-level** (cada caractere é um token) em *Dom Casmurro*, de Machado de Assis
(domínio público, ~400 mil caracteres). É a mesma arquitetura do GPT: embeddings + blocos com atenção causal + MLP.
Só é minúsculo, com ~1,8M de parâmetros contra bilhões.

▶️ **Rode as próximas células antes do intervalo.** O treino continua enquanto vocês tomam café.

In [ ]:
import requests

bruto = requests.get("https://www.gutenberg.org/cache/epub/55752/pg55752.txt").text
inicio = bruto.index("*** START"); inicio = bruto.index("\n", inicio)
texto = bruto[inicio:bruto.index("*** END")].strip()

chars = sorted(set(texto))
stoi = {c: i for i, c in enumerate(chars)}
codificar = lambda s: [stoi[c] for c in s]
decodificar = lambda ids: "".join(chars[i] for i in ids)

dados = torch.tensor(codificar(texto))
n = int(0.9 * len(dados))
treino_ids, val_ids = dados[:n], dados[n:]
print(f"{len(texto):,} caracteres | vocabulário: {len(chars)} símbolos")
print(texto[2000:2300])

In [ ]:
BLOCO, LOTE = 128, 64           # contexto máximo e exemplos por passo
D, CABECAS, CAMADAS = 192, 6, 4

def lote(split):
    fonte = treino_ids if split == "treino" else val_ids
    ix = torch.randint(len(fonte) - BLOCO - 1, (LOTE,))
    x = torch.stack([fonte[i:i + BLOCO] for i in ix])
    y = torch.stack([fonte[i + 1:i + BLOCO + 1] for i in ix])   # alvo = próximo caractere
    return x.to(DEVICE), y.to(DEVICE)

class AutoAtencaoCausal(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(D, 3 * D)
        self.proj = nn.Linear(D, D)
        self.register_buffer("mascara", torch.tril(torch.ones(BLOCO, BLOCO)).view(1, 1, BLOCO, BLOCO))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, CABECAS, C // CABECAS).transpose(1, 2) for t in (q, k, v))
        y, _ = atencao(q, k, v, mascara=self.mascara[:, :, :T, :T])   # a MESMA função da seção 1
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class Bloco(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.atencao = AutoAtencaoCausal()
        self.mlp = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))

    def forward(self, x):
        x = x + self.atencao(self.ln1(x))   # conexão residual
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(len(chars), D)
        self.pos_emb = nn.Embedding(BLOCO, D)
        self.blocos = nn.Sequential(*[Bloco() for _ in range(CAMADAS)])
        self.ln_f = nn.LayerNorm(D)
        self.cabeca = nn.Linear(D, len(chars))

    def forward(self, idx, alvos=None):
        T = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        logits = self.cabeca(self.ln_f(self.blocos(x)))
        perda = None if alvos is None else F.cross_entropy(logits.flatten(0, 1), alvos.flatten())
        return logits, perda

    @torch.no_grad()
    def gerar(self, prompt, n=300, temperatura=1.0):
        idx = torch.tensor([codificar(prompt)], device=DEVICE)
        for _ in range(n):
            logits, _ = self(idx[:, -BLOCO:])
            probs = (logits[:, -1] / temperatura).softmax(-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return decodificar(idx[0].tolist())

gpt = MiniGPT().to(DEVICE)
print(f"{sum(p.numel() for p in gpt.parameters()):,} parâmetros")

In [ ]:
PASSOS = 20 if SMOKE else (2000 if MODO_RAPIDO else 5000)
otimizador = torch.optim.AdamW(gpt.parameters(), lr=2e-3)
agenda = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, PASSOS)
PROMPT = "Capitu "
amostras, historico = {}, []

@torch.no_grad()
def avaliar():
    gpt.eval()
    perdas = {s: torch.stack([gpt(*lote(s))[1] for _ in range(10)]).mean().item() for s in ("treino", "val")}
    gpt.train()
    return perdas

t0 = time.time()
for passo in range(PASSOS + 1):
    if passo in (0, PASSOS // 4, PASSOS):
        gpt.eval(); amostras[passo] = gpt.gerar(PROMPT, n=200); gpt.train()
    if passo % max(PASSOS // 10, 1) == 0:
        perdas = avaliar(); historico.append((passo, perdas["treino"], perdas["val"]))
        print(f"passo {passo:5} | perda treino {perdas['treino']:.3f} | val {perdas['val']:.3f} | {time.time() - t0:.0f}s")
    if passo == PASSOS:
        break
    _, perda = gpt(*lote("treino"))
    otimizador.zero_grad(set_to_none=True)
    perda.backward()
    otimizador.step(); agenda.step()

In [ ]:
for passo, txt in amostras.items():
    print(f"===== passo {passo} =====\n{txt}\n")

h = list(zip(*historico))
plt.figure(figsize=(6, 3))
plt.plot(h[0], h[1], label="treino"); plt.plot(h[0], h[2], label="validação")
plt.xlabel("passo"); plt.ylabel("perda (cross-entropy)"); plt.legend(); plt.title("Curva de aprendizado")

**Temperatura** controla a "criatividade": abaixo de 1 o modelo fica mais conservador e repetitivo, acima de 1 fica mais caótico.

In [ ]:
gpt.eval()
# TODO: gere 200 caracteres a partir de um prompt seu com temperaturas 0.5, 1.0 e 1.5 e compare
...

## 5. Treinar do zero com dados limitados
Em poucos minutos o modelo aprendeu ortografia, pontuação e o "jeito" de Machado, mas **não aprendeu a pensar**.
Com só 400 mil caracteres:
- a perda de **validação** para de cair antes da de treino (*overfitting*);
- o modelo não tem conhecimento de mundo, só padrões do texto.

**Estratégias quando os dados são poucos:** modelo menor, *dropout*, *data augmentation*, parar cedo,
e sobretudo **transfer learning**: partir de um modelo já pré-treinado em bilhões de tokens e só ajustar.
É o tema do Módulo 3.

*Exercício (se der tempo): treine de novo com `treino_ids = dados[:n // 10]` e compare as curvas.*